# AIC211 · Topic 2 · Tuần 5 — TV1 (Trí): Đủ baseline, chốt ngưỡng, E1 + E2

**Mục tiêu:** 5 model baseline có ngưỡng cố định trong `thresholds.json`; có bảng E1 (artifact) và E2 (đổi nguồn).
**Cần có trước:** `A.parquet`, `B.parquet`, `src/common.py` từ tuần 4.

| Bước | Làm gì | Ước tính |
|---|---|---|
| 1–3 | Train LR-F1, RF-F1, XGB-F1, LR-F2, SVM-F2 | 20–40 phút (chủ yếu chờ máy) |
| 4 | Chốt ngưỡng → **báo Nhựt ngay** | 5 phút |
| 5–6 | E2: A-test vs B, confusion matrix, tốc độ | 15 phút |
| 7–9 | E1: chia ngẫu nhiên vs domain, SHAP, bỏ dần feature | 30–45 phút |
| 10 | Viết nháp Dataset & EDA, Method | 1–1,5 giờ |

> **Mốc gấp nhất:** giữa tuần phải xong Ô 4 (`thresholds.json`). Nhựt cần file này để tấn công model. Trễ thì báo Nhựt qua phụ.

In [ ]:
# Ô 1 — Cài thư viện + nối Google Drive (chạy lại mỗi lần mở Colab)
import os, sys, importlib
try:
    import google.colab  # chỉ có khi chạy trên Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install tldextract rapidfuzz shap xgboost
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/AIC211_URL'   # thư mục chung của nhóm
else:
    ROOT = os.path.abspath('AIC211_URL')          # chạy thử trên máy (không phải Colab)

for d in ['data/raw', 'data/processed', 'src', 'models', 'results', 'figures', 'notebooks']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
sys.path.insert(0, f'{ROOT}/src')
print('ROOT =', ROOT)
print(sorted(os.listdir(ROOT)))

In [ ]:
# Ô 2 — Nạp common.py, dữ liệu A/B, feature F1
# Sửa nhỏ common.py (tuần 5): log_result ghi được các thí nghiệm có cột khác nhau. Chạy nhiều lần không sao.
PATCH = """
# --- v2 (tuan 5): log_result chiu duoc cac thi nghiem co cot khac nhau
def log_result(root, exp, model, source, metrics, defense='none', seed=0, k=0, ops='-', mode='-'):
    path = f'{root}/results/results.csv'
    row = {'time': datetime.datetime.now().isoformat(timespec='seconds'), 'exp': exp, 'model': model,
           'defense': defense, 'source': source, 'seed': seed, 'k': k, 'ops': ops, 'mode': mode, **metrics}
    new = pd.DataFrame([row])
    if os.path.exists(path):
        new = pd.concat([pd.read_csv(path), new], ignore_index=True)
    new.to_csv(path, index=False)
"""
_cp = f'{ROOT}/src/common.py'
if '# --- v2' not in open(_cp, encoding='utf-8').read():
    with open(_cp, 'a', encoding='utf-8') as f:
        f.write(PATCH)
import common; importlib.reload(common)
from common import *
import json, time, joblib
import matplotlib.pyplot as plt

A = pd.read_parquet(f'{ROOT}/data/processed/A.parquet')
B = pd.read_parquet(f'{ROOT}/data/processed/B.parquet')
FA, FB = make_F1(A), make_F1(B)
tr, va, te = (A.split == 'train').values, (A.split == 'val').values, (A.split == 'test').values
F1_COLS = list(FA.columns)
print('A:', A.groupby('split').size().to_dict(), '| B:', len(B), '| số feature F1:', len(F1_COLS))

## Bước 3 — Train 5 model
F2 (TF-IDF ký tự) train lâu hơn F1. Lúc chờ thì đọc kết quả EDA tuần 4 để chuẩn bị viết.

In [ ]:
# Ô 3 — Train 5 model baseline trên TRAIN của A (không nhìn B)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

pos_w = float((A.label[tr] == 0).sum() / max((A.label[tr] == 1).sum(), 1))   # cân bằng lớp cho XGB
MODELS = {   # tên: (model, bộ feature)
    'LR_F1':  (make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced')), 'F1'),
    'RF_F1':  (RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1,
                                      class_weight='balanced', random_state=42), 'F1'),
    'XGB_F1': (XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.1, subsample=0.8,
                             colsample_bytree=0.8, scale_pos_weight=pos_w, eval_metric='logloss',
                             random_state=42, n_jobs=-1), 'F1'),
    'LR_F2':  (make_pipeline(make_F2_vectorizer(), LogisticRegression(max_iter=2000, C=4, class_weight='balanced')), 'F2'),
    'SVM_F2': (make_pipeline(make_F2_vectorizer(), LinearSVC(C=0.5, class_weight='balanced')), 'F2'),
}
def X_of(fs, df, F):   # F1 -> bảng số; F2 -> 2 cột chữ host, rest
    return F if fs == 'F1' else df[['host', 'rest']]

for name, (m, fs) in MODELS.items():
    t0 = time.time()
    m.fit(X_of(fs, A[tr], FA[tr]), A.label[tr])
    joblib.dump(m, f'{ROOT}/models/{name}.joblib')
    print(f'{name:7s} train xong trong {time.time() - t0:5.1f}s')

## Bước 4 — Chốt ngưỡng
Ngưỡng mỗi model chọn trên **validation của A** sao cho chỉ ~1% URL sạch bị chặn nhầm. Ghi ra `thresholds.json` rồi **khóa luôn**: mọi thí nghiệm sau (E2, E3, E4) dùng đúng ngưỡng này.

In [ ]:
# Ô 4 — Chốt ngưỡng FPR ≈ 1% trên VAL của A → thresholds.json (ĐÓNG BĂNG, báo Nhựt)
thresholds, info = {}, {}
for name, (m, fs) in MODELS.items():
    s_val = get_scores(m, X_of(fs, A[va], FA[va]))
    thresholds[name] = threshold_at_fpr(A.label[va], s_val, 0.01)
    info[name] = {'features': fs}
json.dump(thresholds, open(f'{ROOT}/models/thresholds.json', 'w'), indent=2)
json.dump(info, open(f'{ROOT}/models/model_info.json', 'w'), indent=2)   # Nhựt cần để biết model dùng F1 hay F2
print(json.dumps({k: round(v, 4) for k, v in thresholds.items()}, indent=1))
print('\n>>> Từ giờ KHÔNG đổi thresholds.json nữa. Muốn đổi phải báo Nhựt.')

## Bước 5–6 — E2: đổi nguồn
Câu hỏi RQ1a: model gần 100% trên ISCX còn bao nhiêu khi gặp PhiUSIIL? `gap_F1` càng lớn thì con số 99% trên benchmark càng ít đáng tin.

In [ ]:
# Ô 5 — E2 (đổi nguồn): metric trên A-test và B, gap = F1(A) − F1(B), confusion matrix
rows = []
for name, (m, fs) in MODELS.items():
    for src, df, F, mask in [('A-test', A, FA, te), ('B', B, FB, np.ones(len(B), bool))]:
        met = evaluate(df.label[mask], get_scores(m, X_of(fs, df[mask], F[mask])), thresholds[name])
        log_result(ROOT, 'E2', name, src, met)
        rows.append({'model': name, 'nguồn': src, **met})
E2_tab = pd.DataFrame(rows)
gap = E2_tab.pivot(index='model', columns='nguồn', values='f1')
gap['gap_F1 (A−B)'] = gap['A-test'] - gap['B']
display(E2_tab[['model', 'nguồn', 'precision', 'recall', 'f1', 'fpr', 'pr_auc']].round(3))
display(gap.round(3))

fig, axes = plt.subplots(2, len(MODELS), figsize=(3 * len(MODELS), 5.6))
for j, name in enumerate(MODELS):
    for i, src in enumerate(['A-test', 'B']):
        r = E2_tab[(E2_tab.model == name) & (E2_tab['nguồn'] == src)].iloc[0]
        cm = np.array([[r.tn, r.fp], [r.fn, r.tp]])
        ax = axes[i, j]; ax.imshow(cm, cmap='Blues')
        for (a, b), v in np.ndenumerate(cm):
            ax.text(b, a, f'{v:,}', ha='center', va='center', fontsize=8)
        ax.set_xticks([0, 1], ['benign', 'độc']); ax.set_yticks([0, 1], ['benign', 'độc'])
        ax.set_title(f'{name} · {src}', fontsize=9)
plt.tight_layout(); plt.savefig(f'{ROOT}/figures/E2_confusion.png', dpi=200); plt.show()

In [ ]:
# Ô 6 — Thời gian dự đoán mỗi URL (tính cả bước trích feature)
sample = A[te].head(1000)
for name, (m, fs) in MODELS.items():
    t0 = time.time()
    F = make_F1(sample) if fs == 'F1' else None
    get_scores(m, X_of(fs, sample, F))
    print(f'{name:7s} {(time.time() - t0) / len(sample) * 1000:.3f} ms/URL')

## Bước 7–9 — E1: artifact
Câu hỏi RQ3: model 99% đang dựa vào cái gì? 3 phép thử: (a) chia ngẫu nhiên có cao hơn chia theo domain không (rò rỉ domain), (b) SHAP chỉ ra feature quan trọng nhất, (c) bỏ dần các feature đó xem F1 tụt bao nhiêu.

In [ ]:
# Ô 7 — E1 (artifact) phần a: chia NGẪU NHIÊN vs chia THEO DOMAIN (RF-F1)
from sklearn.model_selection import train_test_split
rf = lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1,
                                    class_weight='balanced', random_state=42)
idx = np.arange(len(A))
r_tr, r_tmp = train_test_split(idx, test_size=0.30, stratify=A.label, random_state=42)
r_va, r_te = train_test_split(r_tmp, test_size=0.50, stratify=A.label.iloc[r_tmp], random_state=42)
SPLITS = {'random': (r_tr, r_va, r_te), 'domain': (np.where(tr)[0], np.where(va)[0], np.where(te)[0])}

def fit_eval_rf(cols, split):
    a, b, c = SPLITS[split]
    m = rf().fit(FA.iloc[a][cols], A.label.iloc[a])
    thr = threshold_at_fpr(A.label.iloc[b], get_scores(m, FA.iloc[b][cols]))
    return m, evaluate(A.label.iloc[c], get_scores(m, FA.iloc[c][cols]), thr)

f1s = {}
for split in ['random', 'domain']:
    _, met = fit_eval_rf(F1_COLS, split)
    log_result(ROOT, 'E1_split', 'RF_F1', f'A-test-{split}', met)
    f1s[split] = met['f1']
    print(f'chia {split:6s}: F1 = {met["f1"]:.4f} | recall = {met["recall"]:.4f} | FPR = {met["fpr"]:.4f}')
diff = f1s['random'] - f1s['domain']
if diff > 0.01:
    print(f'Chia ngẫu nhiên cao hơn {diff:.3f} -> có dấu hiệu rò rỉ domain (model "nhớ" tên domain). Ghi vào báo cáo.')
else:
    print(f'Chênh lệch {diff:+.3f} (nhỏ) -> chưa thấy rò rỉ domain rõ ràng. Đây vẫn là kết quả hợp lệ, ghi trung thực.')

In [ ]:
# Ô 8 — E1 phần b: SHAP — RF-F1 đang dựa vào feature nào? (chạy trên 1.000 URL cho nhanh)
import shap
rf_main = joblib.load(f'{ROOT}/models/RF_F1.joblib')
Xs = FA[te].sample(min(1000, te.sum()), random_state=0)
sv = shap.TreeExplainer(rf_main).shap_values(Xs)
sv = sv[1] if isinstance(sv, list) else (sv[..., 1] if np.ndim(sv) == 3 else sv)   # lấy lớp "độc"
imp = pd.Series(np.abs(sv).mean(axis=0), index=F1_COLS).sort_values(ascending=False)
imp.to_csv(f'{ROOT}/results/E1_shap_importance.csv')
ax = imp.head(10)[::-1].plot.barh(figsize=(6, 3.5), color='#3b6ea5')
ax.set_xlabel('mean |SHAP|'); ax.set_title('RF-F1: 10 feature quan trọng nhất')
plt.tight_layout(); plt.savefig(f'{ROOT}/figures/E1_shap_top10.png', dpi=200); plt.show()
print('Câu hỏi cho báo cáo: feature đứng đầu có phải dấu hiệu độc thật không, hay chỉ là đặc điểm của cách thu thập dữ liệu?')

In [ ]:
# Ô 9 — E1 phần c: bỏ dần top-k feature quan trọng nhất, train lại, xem F1 tụt bao nhiêu
rows = []
for k in [0, 1, 2, 3, 5]:
    cols = [c for c in F1_COLS if c not in imp.index[:k]]
    for split in ['random', 'domain']:
        _, met = fit_eval_rf(cols, split)
        log_result(ROOT, 'E1_drop', 'RF_F1', f'A-test-{split}', met, k=k, ops='|'.join(imp.index[:k]))
        rows.append({'bỏ top-k': k, 'chia': split, 'f1': met['f1'], 'recall': met['recall']})
E1_tab = pd.DataFrame(rows).pivot(index='bỏ top-k', columns='chia', values='f1')
display(E1_tab.round(4))
print('Đã bỏ theo thứ tự:', list(imp.index[:5]))

## Bước 10 — Viết nháp (vào file báo cáo chung)
- **Dataset & EDA:** bảng 2 nguồn (số URL mỗi lớp), bảng `dedup_log.csv`, 1–2 hình EDA, 3–5 câu về artifact (http/www/path/độ dài khác nhau giữa lớp hoặc nguồn).
- **Method:** F1 (19 feature, liệt kê ngắn), F2 (TF-IDF ký tự 3–5 gram, tách host/path), 5 model, quy tắc ngưỡng FPR 1% chọn trên val A rồi giữ cố định.
- **Kết quả nháp:** bảng E2 + `gap_F1`, hình confusion matrix, bảng E1 + hình SHAP.

## Xong tuần 5 khi
- [ ] `models/*.joblib`, `thresholds.json`, `model_info.json` có đủ 5 model → **đã nhắn Nhựt**
- [ ] Bảng E1, E2 có trong `results/results.csv`; hình trong `figures/`
- [ ] Thứ Năm họp: feature nào đứng đầu SHAP? có dấu hiệu rò rỉ/artifact không?
- [ ] Nháp mục Dataset & EDA, Method